Step 1: Initialize Environment and Load HDFS log datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
from tqdm import tqdm
from collections import Counter

In [ ]:
# Balanced event trace dataset
df = pd.read_csv(
    "/content/drive/MyDrive/Log_Anamoly_Detection/dataset/balanced_log_dataset.csv"
)

# Transformer output (only anomalous rows)
anomaly_results = pd.read_csv(
    "/content/drive/MyDrive/Log_Anamoly_Detection/results/detected_anomalies.csv"
)

# HDFS log templates (EventId -> EventTemplate)
templates_df = pd.read_csv(
    "/content/drive/MyDrive/Log_Anamoly_Detection/dataset/HDFS.log_templates.csv"
)

print("Templates columns:", templates_df.columns.tolist())
print("Logs shape:", df.shape)
print("Anomaly results shape:", anomaly_results.shape)
print(anomaly_results.columns.tolist())

Templates columns: ['EventId', 'EventTemplate']
Logs shape: (33676, 6)
Anomaly results shape: (3368, 5)
['BlockId', 'Predicted_Label', 'Features', 'TimeInterval', 'Latency']


Step 2: Implement Event Mapping and RAG Document Generation Logic

In [ ]:
event_template_map = dict(
    zip(templates_df["EventId"], templates_df["EventTemplate"])
)

# sanity check
event_template_map.get("E5")

'[*]Receiving block[*]src:[*]dest:[*]'

In [ ]:
anomalous_blocks = anomaly_results["BlockId"].unique().tolist()
print("Number of anomalous blocks:", len(anomalous_blocks))

Number of anomalous blocks: 3368


In [ ]:
anomalous_logs = df[df["BlockId"].isin(anomalous_blocks)]

print("Number of anomalous log lines:", len(anomalous_logs))

Number of anomalous log lines: 3368


In [ ]:
from collections import Counter

def generate_rag_document(block_id, row, anomaly_row, template_map):
    # event sequence
    event_sequence = row.get("EventSequence", row.get("Features", []))

    # convert string → list if needed
    if isinstance(event_sequence, str):
        event_sequence = (
            event_sequence.strip("[]")
            .replace("'", "")
            .replace(" ", "")
            .split(",")
        )

    # basic statistics
    total_events = len(event_sequence)
    event_counts = Counter(event_sequence)

    # frequently repeated events (important for RCA)
    frequent_events = {
        eid: count
        for eid, count in event_counts.items()
        if count > 2
    }

    # map ONLY important EventIds to meanings
    key_event_meanings = {
        eid: template_map.get(eid, "Unknown event")
        for eid in frequent_events.keys()
    }

    latency = anomaly_row.get("Latency", "N/A")
    time_interval = anomaly_row.get("TimeInterval", "N/A")

    rag_text = f"""
Block ID:
{block_id}

Summary:
This HDFS execution trace was flagged as anomalous due to repeated system events
and irregular execution timing.

Event Statistics:
- Total events: {total_events}
- Unique event types: {len(set(event_sequence))}
- Frequently repeated events: {frequent_events}

Key Event Meanings:
"""

    for eid, meaning in key_event_meanings.items():
        rag_text += f"- {eid}: {meaning}\n"

    rag_text += f"""

Latency Information:
- Observed latency: {latency}

Timing Behavior:
- Time intervals indicate irregular or delayed execution.

Likely Root Cause Indicators:
- Repeated block receive and packet termination events
- Frequent add/delete block operations
- Elevated end-to-end latency

Context:
This anomaly was detected by a Transformer-based sequence model trained on HDFS
event traces. Such patterns often indicate retries, unstable block replication,
or transient communication failures in distributed storage systems.
"""

    return rag_text

Step 3: Execute Batch Document Generation and Export Results

In [ ]:
rag_documents = []

filtered_df = df[df["BlockId"].isin(anomalous_blocks)]

for _, row in tqdm(filtered_df.iterrows(), total=len(filtered_df)):
    block_id = row["BlockId"]

    anomaly_row = anomaly_results[
        anomaly_results["BlockId"] == block_id
    ].iloc[0]

    doc_text = generate_rag_document(
        block_id, row, anomaly_row, event_template_map
    )

    rag_documents.append({
        "BlockId": block_id,
        "Document": doc_text
    })

rag_df = pd.DataFrame(rag_documents)

print("RAG documents created:", len(rag_df))


100%|██████████| 3368/3368 [00:04<00:00, 777.80it/s]

RAG documents created: 3368


In [ ]:
rag_df.to_csv("/content/drive/MyDrive/Log_Anamoly_Detection/results/rag_documents.csv", index=False)

In [ ]:
print(rag_df.iloc[0]["Document"])


Block ID:
blk_8462687553742484299

Summary:
This HDFS execution trace was flagged as anomalous due to repeated system events
and irregular execution timing.

Event Statistics:
- Total events: 20
- Unique event types: 8
- Frequently repeated events: {'E5': 3, 'E11': 3, 'E9': 3, 'E26': 3, 'E23': 3, 'E21': 3}

Key Event Meanings:
- E5: [*]Receiving block[*]src:[*]dest:[*]
- E11: [*]PacketResponder[*]for block[*]terminating[*]
- E9: [*]Received block[*]of size[*]from[*]
- E26: [*]BLOCK* NameSystem[*]addStoredBlock: blockMap updated:[*]is added to[*]size[*]
- E23: [*]BLOCK* NameSystem[*]delete:[*]is added to invalidSet of[*]
- E21: [*]Deleting block[*]file[*]


Latency Information:
- Observed latency: 7156

Timing Behavior:
- Time intervals indicate irregular or delayed execution.

Likely Root Cause Indicators:
- Repeated block receive and packet termination events
- Frequent add/delete block operations
- Elevated end-to-end latency

Context:
This anomaly was detected by a Transformer-base